In [1]:
# preprocess_and_shard_test.py
# Run once to produce 3 shards from the test set without removing any data.
import os
import numpy as np
import pandas as pd

# Set seeds for reproducibility if any random operations were needed
np.random.seed(42)

# --- Configuration ---
# !! IMPORTANT: Update this to the path of your test.csv file !!
IN = "/kaggle/input/test1-csv/test.csv"
OUT_DIR = "/kaggle/working/"
os.makedirs(OUT_DIR, exist_ok=True)

# --- Load Data ---
df = pd.read_csv(IN)
N_original = len(df)
print(f"Original rows in {os.path.basename(IN)}: {N_original}")

# --- Preprocessing ---
# 1) Basic cleaning: Remove rows with invalid or missing prices (if 'price' column exists)
# This step is often unnecessary for test sets but is good practice.
if 'price' in df.columns:
    df = df[~df['price'].isna()].copy()
    df = df[df['price'] >= 0.0]
df = df.reset_index(drop=True)
print(f"Rows after basic cleaning: {len(df)}")

# The percentile trimming and stratified sampling sections have been removed.
# We will use the entire cleaned dataframe for sharding.
df_processed = df

print(f"Total rows to be sharded: {len(df_processed)}")

# --- Sharding ---
# 2) Create 3 nearly equal shards from the full dataset
k = 9
rows = len(df_processed)
size = rows // k
print(f"Creating {k} shards of approximate size {size}...")

for i in range(k):
    start = i * size
    # Ensure the last shard captures all remaining rows
    end = (i + 1) * size if i < k - 1 else rows

    shard = df_processed.iloc[start:end].reset_index(drop=True)
    out_path = os.path.join(OUT_DIR, f"shard_{i}.csv")
    shard.to_csv(out_path, index=False)
    print(f"Saved {out_path} with {len(shard)} rows.")

# --- Verification ---
total_saved = sum(len(pd.read_csv(os.path.join(OUT_DIR, f"shard_{i}.csv"))) for i in range(k))
print("\nDone.")
print(f"Total rows saved across all shards: {total_saved}")

if total_saved == len(df_processed):
    print("Verification successful: Total saved rows match processed rows. ✅")
else:
    print("Verification failed: Mismatch in row counts. ❌")

Original rows in test.csv: 75000
Rows after basic cleaning: 75000
Total rows to be sharded: 75000
Creating 9 shards of approximate size 8333...
Saved /kaggle/working/shard_0.csv with 8333 rows.
Saved /kaggle/working/shard_1.csv with 8333 rows.
Saved /kaggle/working/shard_2.csv with 8333 rows.
Saved /kaggle/working/shard_3.csv with 8333 rows.
Saved /kaggle/working/shard_4.csv with 8333 rows.
Saved /kaggle/working/shard_5.csv with 8333 rows.
Saved /kaggle/working/shard_6.csv with 8333 rows.
Saved /kaggle/working/shard_7.csv with 8333 rows.
Saved /kaggle/working/shard_8.csv with 8336 rows.

Done.
Total rows saved across all shards: 75000
Verification successful: Total saved rows match processed rows. ✅
